# 08. Turn a biological question into a test

We will connect neurological and muscle limitations to observable movement, count all current GAVD folders, and freeze source roles without using model performance.

## Run this notebook on its own

From `experiments/aaai`, run `uv sync`, then `uv run jupyter lab`. Select the Python environment created here. Python 3.12, PyTorch, pandas, scikit-learn, matplotlib, python-dotenv, the notebook libraries, and the local `ambient` package are declared in `pyproject.toml`. Each notebook imports and loads everything it needs.

All configuration and credentials remain in the repository root `.env`. Set `AAAI_DEMO=1` for artificial arrays used to check the software, or `AAAI_DEMO=0` for available real poses. Restart the kernel after changing settings. Artificial examples have no medical meaning. The setup uses aliases for python-dotenv's actual function names so that `load_env(find_env())` works.

The video below provides real walking context in either mode. It is separate from any artificial example. These research notebooks do not launch a large download or full experiment without an explicit command.

## Historical tutorial and current study

This notebook preserves the earlier tutorial's code, observations and experimental scope. Its original result identities remain historical; they are not measurements of the new retained-predictor or U/J/A study. Start the current independent tutorial sequence at [Notebook 12](12_observation_population_and_video_context.ipynb), then continue through [Notebook 17](17_failure_analysis_and_manuscript_decision.ipynb). The [current protocol](docs/iteration02/protocol.md) and [current results](docs/iteration02/results.md) identify the verified new run. The [draft01 manuscript](docs/draft01/manuscript.md) remains the record of the completed earlier study.

The current primary question is normal versus the four requested condition families, with all other folders retained in a mandatory broad secondary analysis. It tests opposite-leg information beyond own-leg geometry and extra capacity. Uniform masking remains the reference, while J and A are declared motion-independent alternatives with matched budgets. Read the newer notebooks for their support rules and fixed evaluation banks. Do not combine old and new scores or denominators.

In [ ]:
from pathlib import Path
import os, sys, json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'ambient').is_dir() and (p/'data/gavd').is_dir())
BASE=ROOT/'experiments/aaai'
os.chdir(ROOT)
from dotenv import load_dotenv as load_env, find_dotenv as find_env
load_env(find_env())
os.chdir(BASE)
if str(BASE) not in sys.path:sys.path.insert(0,str(BASE))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
from aaai_gait.settings import Settings
from aaai_gait.study_data import build_study_manifest,make_outer_folds,fold_metadata
from aaai_gait.study_runner import prepare_study
from aaai_gait.media import show_video_context,plot_motion
settings=Settings();torch.set_num_threads(2)
fresh_manifest=build_study_manifest(ROOT,settings.pose_dir)
print('Mode:', 'SYNTHETIC SOFTWARE CHECK' if settings.demo else 'REAL GAVD, subject to quality exclusions')
print('Fresh annotations:',len(fresh_manifest),'sequences;',fresh_manifest.source_id.nunique(),'source videos')
video_row=show_video_context(fresh_manifest,frame_number_base=settings.frame_number_base)

## 1. Start with movement that the video can support

After stroke, a person may move one leg differently to maintain forward progress. That pattern can be highly repeatable. Parkinson's disease can affect movement size and timing differently. Cerebral palsy covers varied developmental motor impairments, and visible walking patterns change with growth and muscle properties. Myopathic gait describes a family of muscle-related problems. It is not one disease of the brain.

These observations motivate checking movement relationships. They do not let us infer a lesion, dopamine level, muscle force, or diagnosis from a web video. The [neurological review](docs/draft00/neurological_research_direction.md) links the primary clinical studies behind each hypothesis.

![Biology, movement, and camera observations](docs/draft00/images/neurological_chain.svg)

![Different causes can produce overlapping movements](docs/draft00/images/condition_routes.svg)

## 2. Ask a question with a concrete comparison

Hide four consecutive positions of the left ankle. First predict them from the ankle's other visible positions. Then predict the same positions with the other visible joints available. The difference in prediction error measures how much the extra context helped this particular predictor. We repeat the comparison over random masks and twelve readout joints.

Both reduced and increased help can be unusual. A compensation may make two body parts move together more tightly. A scalar error can also be low for an abnormal but predictable movement. Our proposed three-region profile preserves this possibility without choosing a score direction after viewing the test results.

![An example of predictive help](docs/draft00/images/predictive_benefit.svg)

The task is masked completion. Visible positions may come before and after a hidden section. Notebook 05 separately teaches forecasting with a fully hidden future.

## Watch an annotated abnormal example

The dataset's condition label provides context, not a confirmed diagnosis. This example previews a stroke-labeled sequence. Compare the visible movement with the normal preview above without assuming which side is affected. Set `AAAI_PREVIEW_LABEL` in the root `.env` to `parkinsons`, `cerebral palsy`, or `myopathic` to inspect another family.

In [ ]:
condition_to_view=os.getenv('AAAI_PREVIEW_LABEL','stroke')
condition_video=show_video_context(fresh_manifest,label=condition_to_view,frame_number_base=settings.frame_number_base)

In [ ]:
illustration=pd.DataFrame({'situation':['Familiar relationship','Less help','More help'],
                          'own_error':[.20,.20,.20],'full_error':[.08,.18,.03]})
illustration['help']=illustration.own_error-illustration.full_error
print('Invented teaching values, not measured GAVD results.')
display(illustration)

## 3. Use all folders for the binary question

Only `normal` is normal. Every other current condition folder is abnormal for this experiment, including `exercise`, `style`, and `pregnant`. These labels describe the task and are not all medical diagnoses. The four condition families remain the focus of biological discussion and the separate five-class classifier. Stroke appears once.

Read the current CSVs each time. The `source` directory is a distribution cache and must not add duplicate observations. A source video can contain several sequences, and a sequence can provide overlapping windows. These are not additional independent participants.

In [ ]:
census=fresh_manifest.groupby('label').agg(sequences=('sequence_id','size'),rows=('n_frames','sum'),sources=('source_id','nunique'))
display(census)
assert fresh_manifest.loc[fresh_manifest.label.eq('normal'),'binary_label'].eq(0).all()
assert fresh_manifest.loc[fresh_manifest.label.ne('normal'),'binary_label'].eq(1).all()
print('Manifest fingerprint:',fresh_manifest.attrs['manifest_sha256'])

## 4. Retrieve media, extract poses, and record failures

Run the following command in a terminal when ready. The fresh inventory covers all twelve folders. Downloads go to `data/videos-gavd/<source_id>/`. Source IDs avoid duplicate downloads when a video has several annotated sequences. Existing decodable local media are reused.

```bash
uv run python scripts/ingest_study.py --download-missing --download-budget-seconds 900 --source-timeout 45 --download-workers 4 --workers 4
```

A fixed acquisition budget prevents a few unavailable videos from blocking the study. Repeat the command to continue acquisition. Logs record videos not attempted, timeouts, and failures. The pose pipeline uses `ambient` through the annotated-person crop adapter, retains raw confidence and frame indices, and hashes the annotation, video, model, and extraction settings. Fixed observation-quality rules apply before folds. A score-dependent numerical failure must raise an error rather than become a quiet exclusion.

![Pipeline and controls](docs/draft00/images/neuromechanics_experiment.svg)

In [ ]:
x,meta,manifest,ledger=prepare_study(demo=settings.demo,seed=settings.seed)
print('Available QC-passing windows:',x.shape)
display(meta.groupby('label').agg(sources=('source_id','nunique'),sequences=('sequence_id','nunique'),windows=('sequence_id','size')))

## 5. Separate fitting, reference estimation, and testing

Four outer folds distribute normal sources and abnormal category memberships without seeing scores. In each fold, five remaining normal sources estimate the normal profile reference. Other normal sources fit the encoder and predictors. Non-test abnormal sources are auxiliary for the separate supervised classifier only. They never fit the normal-only anomaly model.

Before assigning folds, the full runner also checks that every source has four distinct eligible physical patches for each readout joint. This check reads masks and observation validity, not losses. Repeated masks and overlapping windows do not create additional physical support.

In [ ]:
from aaai_gait.predictive_profile import freeze_profile_eligibility
from aaai_gait.context_prediction import ContextConfig
keep,eligibility=freeze_profile_eligibility(x,meta,config=ContextConfig())
eligible_meta=meta.loc[keep].reset_index(drop=True)
registry=make_outer_folds(eligible_meta,seed=settings.seed)
roles=fold_metadata(eligible_meta,registry,outer_fold=0)
display(registry.groupby(['outer_fold','binary_label']).size().unstack(fill_value=0))
display(roles.groupby('split').agg(sources=('source_id','nunique'),windows=('source_id','size')))
assert roles.loc[roles.split.isin(['train','val']),'label'].eq('normal').all()

## 6. Review visible movement without making a diagnosis

A small blinded review should describe whether the person is walking, the camera view, tracking identity, visible asymmetry, arm excursion, knee flexion, trunk motion, and aids. Keep the label lookup separate. Uncertain observations remain uncertain. Two independent reviewers and agreement reporting would strengthen the biological interpretation. A blank template is not evidence that review occurred.

In [ ]:
from aaai_gait.study_controls import movement_annotation_template
blind,lookup=movement_annotation_template(fresh_manifest,per_label=2,seed=settings.seed)
display(blind.head())
print('Blank review rows:',len(blind),'; label lookup is a separate table.')

## Record what this experiment establishes

Keep the run protocol, source roles, exclusions, training curves, frozen reference parameters, and source scores together. A diagram explains an intended comparison. Only a completed real-data run supplies evidence for it. Read the [implemented protocol](docs/context_study_protocol.md), [current results](docs/context_study_results.md), and [review of the research direction](docs/draft00/README.md) before making a manuscript claim.